# Taller N°1 — Consumo de APIs

**Asignatura:** Machine Learning — MLY1101 (Sección 001V)  
**Estudiante:** Hernán Lippke  
**Objetivo:** Extraer datos automáticamente desde 3 APIs públicas, generar 3 archivos de datos independientes y publicar el trabajo en GitHub.

> **Reproducibilidad:** el notebook está pensado para ejecutarse con *Entorno de ejecución → Ejecutar todas* y regenerar los 3 datasets sin intervención manual. Las 3 APIs son **sin API key**.

## 1. Pregunta u objetivo común

> **¿Qué información puede recopilarse sobre los países del mundo desde fuentes geográfico-demográficas, económicas y culturales?**

Cada fuente aporta a esta pregunta desde un ángulo distinto.  
**No se busca responder la pregunta ni integrar, unir o cruzar las fuentes.** Cada API se consume y se guarda de forma independiente.

## 2. Fuentes seleccionadas

Tres APIs públicas de la lista `public-apis/public-apis`, todas **sin API key**:

| # | API | Aporta a la pregunta | Enlace |
|---|-----|----------------------|--------|
| 1 | **REST Countries** | Geografía y demografía (capital, región, población, área, idiomas, monedas, coordenadas) | https://restcountries.com/ |
| 2 | **World Bank** | Economía (indicador PIB per cápita por país) | https://data.worldbank.org/ |
| 3 | **Nager.Date** | Cultura (días festivos oficiales por país) | https://date.nager.at/ |

## 3. Configuración común

Imports, carpeta de salida y utilidades compartidas por las 3 descargas.

In [1]:
import os, time, json
import requests
import pandas as pd

OUT_DIR = "."            # misma carpeta del repositorio
MIN_RECORDS = 200        # objetivo mínimo de registros por API
os.makedirs(OUT_DIR, exist_ok=True)

def guardar_csv(registros, nombre):
    """Guarda una lista de dicts como CSV y reporta el número de registros."""
    df = pd.DataFrame(registros)
    ruta = os.path.join(OUT_DIR, nombre)
    df.to_csv(ruta, index=False, encoding="utf-8")
    print(f"{nombre}: {len(df)} registros -> {ruta}")
    return df

print("Configuración lista.")

Configuración lista.


## 3.1 API 1 — CountriesNow (demografía y geografía)

- **Endpoint:** `https://countriesnow.space/api/v0.1/countries/population`
- **Autenticación:** ninguna (API abierta, sin key).
- **Registros objetivo:** ≥ 200 (una llamada devuelve ~240 países).
- **Paginación:** no es necesaria; una sola llamada trae todos los países.
- **Salida:** `dataset_api_1.csv`

In [5]:
# =============================================================================
# API 1 — CountriesNow: población de los países (demografía)
# -----------------------------------------------------------------------------
# LÓGICA GENERAL:
#   1) Se consulta el endpoint /countries/population, que devuelve TODOS los
#      países con su historial de población. Es una API abierta (sin key).
#   2) Se valida el PAYLOAD (campo 'error' y lista 'data'), no solo el status,
#      para no dar por buena una respuesta de error con código 200.
#   3) Cada país trae 'populationCounts' (lista por año); se APLANA tomando el
#      año más reciente -> una fila por país.
#   4) Una sola llamada supera los 200 registros, así que NO requiere paginación.
#   5) Se arma el DataFrame y se guarda como dataset_api_1.csv.
# =============================================================================

# --- Constantes de la fuente ---
CN_URL = "https://countriesnow.space/api/v0.1/countries/population"


def descargar_poblacion(reintentos=3, espera=5):
    """
    Descarga los datos de población de todos los países desde CountriesNow.
    Devuelve: la lista 'data' (países), o [] si falla o el payload es inválido.
    """
    for intento in range(1, reintentos + 1):
        try:
            resp = requests.get(CN_URL,
                                headers={"User-Agent": "TallerMLY1101-countriesnow/1.0"},
                                timeout=60)
            if resp.status_code == 429:
                print(f"   [429] límite de tasa; esperando {espera*intento}s...")
                time.sleep(espera * intento)
                continue
            resp.raise_for_status()
            data = resp.json()
            # Validación de contenido: 'error' debe ser False y 'data' una lista.
            if isinstance(data, dict) and data.get("error") is False and isinstance(data.get("data"), list):
                return data["data"]
            print("   payload inesperado:", str(data)[:200])
            return []
        except requests.RequestException as e:
            print(f"   Error de conexión ({type(e).__name__}); reintento {intento}/{reintentos}")
            time.sleep(espera * intento)
    return []


def aplanar_poblacion(item):
    """
    Convierte un país (con historial de población) en una fila:
    toma el registro del AÑO MÁS RECIENTE de 'populationCounts'.
    """
    counts = item.get("populationCounts") or []
    # max por año; si no hay conteos, dict vacío (poblacion/anio quedarán en None).
    ultimo = max(counts, key=lambda c: int(c.get("year", 0))) if counts else {}
    return {
        "pais":      item.get("country"),
        "code":      item.get("code"),      # ISO2
        "iso3":      item.get("iso3"),      # ISO3
        "poblacion": ultimo.get("value"),
        "anio":      ultimo.get("year"),
    }


# --- 1) Descargar ---
paises = descargar_poblacion()
print(f"Países recibidos: {len(paises)}")

# --- 2) Aplanar cada país a una fila ---
registros = [aplanar_poblacion(p) for p in paises]

# --- 3) DataFrame + guardado (con resguardo si vino vacío) ---
df_api1 = pd.DataFrame(registros)

if len(df_api1) == 0:
    print("\nADVERTENCIA: CountriesNow no devolvió datos en esta corrida. "
        "No se sobrescribe dataset_api_1.csv; reintenta esta celda en unos minutos.")
else:
    guardar_csv(registros, "dataset_api_1.csv")
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", None)
    print(f"\nRegistros totales: {len(df_api1)}")
    print(f"Columnas ({len(df_api1.columns)}): {list(df_api1.columns)}")

df_api1   # última expresión: muestra la tabla en pantalla

Países recibidos: 263
dataset_api_1.csv: 263 registros -> .\dataset_api_1.csv

Registros totales: 263
Columnas (5): ['pais', 'code', 'iso3', 'poblacion', 'anio']


,pais,code,iso3,poblacion,anio
0,Arab World,ARB,ARB,419790588,2018
1,Caribbean small states,CSS,CSS,7358965,2018
2,Central Europe and the Baltics,CEB,CEB,102511922,2018
3,Early-demographic dividend,EAR,EAR,3249140605,2018
4,East Asia & Pacific,EAS,EAS,2328220870,2018
...,...,...,...,...,...
258,Virgin Islands (U.S.),VIR,VIR,106977,2018
259,West Bank and Gaza,PSE,PSE,4569087,2018
260,"Yemen, Rep.",YEM,YEM,28498687,2018
261,Zambia,ZMB,ZMB,17351822,2018


## 3.2 API 2 — World Bank (economía)

- **Endpoint:** `https://api.worldbank.org/v2/country/all/indicator/NY.GDP.PCAP.CD?format=json`
- **Autenticación:** ninguna.
- **Registros objetivo:** ≥ 200.
- **Paginación:** la API pagina los resultados (`page` / `per_page`); se recorren las páginas y se acumulan → aquí se implementa la paginación.
- **Salida:** `dataset_api_2.csv`

In [7]:
# =============================================================================
# API 2 — World Bank: PIB per cápita por país (economía)
# -----------------------------------------------------------------------------
# LÓGICA GENERAL:
#   1) World Bank devuelve una LISTA de 2 elementos: [ metadata, [registros] ].
#      La metadata trae 'pages' (número total de páginas).
#   2) PAGINACIÓN REAL: se pide la página 1, se lee cuántas páginas hay y se
#      recorren todas (page=1..pages), acumulando los registros de cada una.
#   3) Cada registro se APLANA a columnas simples (país, iso3, año, valor).
#   4) Se arma el DataFrame y se guarda como dataset_api_2.csv.
# =============================================================================

# --- Constantes de la fuente ---
WB_URL = "https://api.worldbank.org/v2/country/all/indicator/NY.GDP.PCAP.CD"
WB_PER_PAGE = 100     # registros por página
WB_ANIO = "2022"      # año del indicador (PIB per cápita)


def descargar_pagina_wb(page, reintentos=3, espera=5):
    """
    Descarga UNA página del indicador. Devuelve la tupla (metadata, registros).
    Valida la estructura [metadata, registros] antes de devolver.
    """
    params = {"format": "json", "per_page": WB_PER_PAGE, "page": page, "date": WB_ANIO}
    for intento in range(1, reintentos + 1):
        try:
            resp = requests.get(WB_URL, params=params,
                                headers={"User-Agent": "TallerMLY1101-worldbank/1.0"},
                                timeout=60)
            if resp.status_code == 429:
                print(f"   [429] límite de tasa; esperando {espera*intento}s...")
                time.sleep(espera * intento)
                continue
            resp.raise_for_status()
            data = resp.json()
            # World Bank responde [metadata, [registros]]; validamos el contenido.
            if isinstance(data, list) and len(data) == 2 and isinstance(data[1], list):
                return data[0], data[1]
            print("   payload inesperado:", str(data)[:200])
            return {}, []
        except requests.RequestException as e:
            print(f"   Error de conexión ({type(e).__name__}); reintento {intento}/{reintentos}")
            time.sleep(espera * intento)
    return {}, []


def aplanar_wb(r):
    """Convierte un registro de World Bank en una fila plana."""
    return {
        "pais":           r.get("country", {}).get("value"),
        "iso3":           r.get("countryiso3code"),
        "anio":           r.get("date"),
        "pib_per_capita": r.get("value"),   # puede ser None si el país no reporta
    }


# --- 1) Primera página: además de datos, nos dice cuántas páginas hay ---
meta, registros = descargar_pagina_wb(1)
total_paginas = int(meta.get("pages", 1)) if meta else 1
print(f"Total de páginas a recorrer: {total_paginas}")

acumulado = list(registros)   # arrancamos con los registros de la página 1

# --- 2) PAGINACIÓN: recorrer de la página 2 hasta la última ---
for page in range(2, total_paginas + 1):
    print(f"Descargando página {page}/{total_paginas} ...")
    _, registros_pagina = descargar_pagina_wb(page)
    acumulado.extend(registros_pagina)
    time.sleep(1)             # pausa breve entre páginas
    print(f"   acumulado: {len(acumulado)} registros")

# --- 3) Aplanar cada registro a una fila ---
filas = [aplanar_wb(r) for r in acumulado]

# --- 4) DataFrame + guardado (con resguardo si vino vacío) ---
df_api2 = pd.DataFrame(filas)

if len(df_api2) == 0:
    print("\nADVERTENCIA: World Bank no devolvió datos en esta corrida. "
        "No se sobrescribe dataset_api_2.csv; reintenta esta celda en unos minutos.")
else:
    guardar_csv(filas, "dataset_api_2.csv")
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", None)
    print(f"\nRegistros totales: {len(df_api2)}")
    print(f"Columnas ({len(df_api2.columns)}): {list(df_api2.columns)}")

df_api2   # última expresión: muestra la tabla en pantalla

Total de páginas a recorrer: 3
Descargando página 2/3 ...
   acumulado: 200 registros
Descargando página 3/3 ...
   acumulado: 265 registros
dataset_api_2.csv: 265 registros -> .\dataset_api_2.csv

Registros totales: 265
Columnas (4): ['pais', 'iso3', 'anio', 'pib_per_capita']


,pais,iso3,anio,pib_per_capita
0,Africa Eastern and Southern,AFE,2022,1675.902524
1,Africa Western and Central,AFW,2022,2143.072094
2,Arab World,ARB,2022,7960.811588
3,Caribbean small states,CSS,2022,17449.519342
4,Central Europe and the Baltics,CEB,2022,19526.246344
...,...,...,...,...
260,Virgin Islands (U.S.),VIR,2022,44320.909186
261,West Bank and Gaza,PSE,2022,3799.955270
262,"Yemen, Rep.",YEM,2022,NaN
263,Zambia,ZMB,2022,1447.123101


## 3.3 API 3 — Nager.Date (cultura: días festivos)

- **Endpoints:** `.../v3/AvailableCountries` y `.../v3/PublicHolidays/{año}/{país}`
- **Autenticación:** ninguna, sin límite de tasa.
- **Registros objetivo:** ≥ 200 (se recorren los países disponibles y se acumulan sus feriados).
- **Salida:** `dataset_api_3.csv`

In [9]:
# =============================================================================
# API 3 — Nager.Date: días festivos oficiales por país (cultura)
# -----------------------------------------------------------------------------
# LÓGICA GENERAL:
#   1) Se pide la lista de países disponibles (AvailableCountries).
#   2) Se RECORRE país por país pidiendo sus feriados de un año fijo
#      (PublicHolidays/{año}/{país}). Esa iteración es la "paginación" natural
#      de esta fuente: una petición por país, acumulando los resultados.
#   3) A cada feriado se le agrega el nombre del país y se aplana a una fila.
#   4) Se corta cuando ya se supera con holgura el mínimo de 200 registros
#      (no hace falta recorrer los 204 países).
#   5) Se arma el DataFrame y se guarda como dataset_api_3.csv.
# =============================================================================

# --- Constantes de la fuente ---
NAGER_BASE = "https://date.nager.at/api/v3"
NAGER_ANIO = 2024          # año fijo -> reproducible (feriados históricos estables)
OBJETIVO   = 300           # meta cómoda por encima de los 200 exigidos


def consultar_nager(url, reintentos=3, espera=3):
    """GET a un endpoint de Nager.Date. Devuelve la lista JSON, o [] si falla."""
    for intento in range(1, reintentos + 1):
        try:
            resp = requests.get(url, headers={"User-Agent": "TallerMLY1101-nager/1.0"},
                                timeout=30)
            resp.raise_for_status()
            data = resp.json()
            return data if isinstance(data, list) else []
        except requests.RequestException as e:
            print(f"   Error ({type(e).__name__}) en {url}; reintento {intento}/{reintentos}")
            time.sleep(espera * intento)
    return []


def aplanar_feriado(pais_nombre, h):
    """Convierte un feriado en una fila plana, agregando el nombre del país."""
    tipos = h.get("types") or []      # 'types' es una lista (ej. ['Public'])
    return {
        "pais":         pais_nombre,
        "countryCode":  h.get("countryCode"),
        "fecha":        h.get("date"),
        "nombre_local": h.get("localName"),
        "nombre_en":    h.get("name"),
        "tipo":         ", ".join(tipos) if tipos else None,
        "global":       h.get("global"),
        "fijo":         h.get("fixed"),
    }


# --- 1) Lista de países disponibles ---
paises = consultar_nager(f"{NAGER_BASE}/AvailableCountries")
print(f"Países disponibles: {len(paises)}")

# --- 2) Recorrer países acumulando sus feriados (iteración = paginación) ---
registros = []
for pais in paises:
    codigo = pais.get("countryCode")
    nombre = pais.get("name")
    feriados = consultar_nager(f"{NAGER_BASE}/PublicHolidays/{NAGER_ANIO}/{codigo}")
    for h in feriados:
        registros.append(aplanar_feriado(nombre, h))
    time.sleep(0.2)                       # pausa mínima (la API no exige límite)
    # Nos detenemos al superar la meta: no hace falta recorrer los 204 países.
    if len(registros) >= OBJETIVO:
        print(f"   objetivo alcanzado: {len(registros)} feriados "
            f"(hasta {nombre}); se detiene el recorrido.")
        break

# --- 3) DataFrame + guardado (con resguardo si vino vacío) ---
df_api3 = pd.DataFrame(registros)

if len(df_api3) == 0:
    print("\nADVERTENCIA: Nager.Date no devolvió datos en esta corrida. "
        "No se sobrescribe dataset_api_3.csv; reintenta esta celda en unos minutos.")
else:
    guardar_csv(registros, "dataset_api_3.csv")
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", None)
    print(f"\nRegistros totales: {len(df_api3)}")
    print(f"Columnas ({len(df_api3.columns)}): {list(df_api3.columns)}")

df_api3   # última expresión: muestra la tabla en pantalla

Países disponibles: 204
   objetivo alcanzado: 310 feriados (hasta Bolivia); se detiene el recorrido.
dataset_api_3.csv: 310 registros -> .\dataset_api_3.csv

Registros totales: 310
Columnas (8): ['pais', 'countryCode', 'fecha', 'nombre_local', 'nombre_en', 'tipo', 'global', 'fijo']


,pais,countryCode,fecha,nombre_local,nombre_en,tipo,global,fijo
0,Andorra,AD,2024-01-01,Any nou,New Year's Day,Public,True,False
1,Andorra,AD,2024-01-06,Reis,Epiphany,Public,True,False
2,Andorra,AD,2024-02-12,Carnaval,Carnival,Public,True,False
3,Andorra,AD,2024-03-14,Dia de la Constitució,Constitution Day,Public,True,False
4,Andorra,AD,2024-03-29,Divendres Sant,Good Friday,Public,True,False
...,...,...,...,...,...,...,...,...
305,Bolivia,BO,2024-06-21,Año Nuevo Andino,Andean New Year,Public,True,False
306,Bolivia,BO,2024-08-02,Día de la Revolución Agraria,Agrarian Reform Day,Public,True,False
307,Bolivia,BO,2024-08-06,Dia de la Patria,Independence Day,Public,True,False
308,Bolivia,BO,2024-11-02,Todos Santos,All Saints' Day,Public,True,False
